# vLLM foundations: KV cache and continuous batching

## Learning goals
- Compute KV-cache memory and see block paging fight fragmentation
- Contrast static vs continuous batching on the same workload
- Understand what AR stages inherit for free from vLLM

> **Mac track.** Every executed cell below is a pure-Python simulation that runs on any Mac with no GPU. Cells labelled **Linux GPU lab** show the *real* vLLM-Omni commands — they are shown as text and are **not** run here, because the official `vllm serve --omni` runtime is CUDA-only.

## Why AR generation is memory-bound

Every generated token must attend to all previous tokens, so their keys and values are cached. That KV cache grows with sequence length and dominates accelerator memory — which is what caps how many requests run at once. vLLM's **PagedAttention** carves the cache into fixed **blocks** and allocates them lazily, so sequences share memory instead of pre-reserving their maximum length.

In [ ]:
from omni_tutorial import kv_bytes_per_token, PagedKVCache
# A small model's per-token footprint across all layers:
per_tok = kv_bytes_per_token(num_layers=28, num_kv_heads=4, head_dim=128, dtype='fp16')
print(f'{per_tok:,} bytes/token  ({per_tok/1024:.1f} KiB)')
seq = 2048
print(f'one 2048-token sequence: {per_tok*seq/1e6:.1f} MB of KV cache')

## Paging and fragmentation

We grow a few sequences token-by-token in a fixed block pool and watch utilization. The gap between allocated and used slots is *internal fragmentation* — wasted space in each sequence's partially-filled last block.

In [ ]:
cache = PagedKVCache(total_blocks=64, block_size=16)
import itertools
seqs = ['a', 'b', 'c']
used, wasted = [], []
for step in range(40):
    for s in seqs:
        cache.append(s, 1)
    used.append(cache.used_slots)
    wasted.append(cache.wasted_slots)
print(f'after 40 steps: {cache.used_slots} slots used, {cache.wasted_slots} wasted, '
      f'utilization {cache.utilization:.0%}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
fig, ax = plt.subplots(figsize=(6, 3.2))
steps = range(1, len(used) + 1)
ax.stackplot(steps, used, wasted, labels=['used slots', 'wasted (fragmentation)'],
             colors=['#4c78a8', '#f2cf5b'])
ax.set_xlabel('decode step'); ax.set_ylabel('KV slots')
ax.set_title('Paged KV cache: used vs fragmented slots'); ax.legend(loc='upper left')
fig.tight_layout(); plt.show()

## Static vs continuous batching

One long request shares a batch with several short ones. Static batching locks the batch until the straggler finishes; continuous batching refills freed slots immediately.

In [ ]:
from omni_tutorial import (Request, simulate_static_batching,
                           simulate_continuous_batching, throughput)
reqs = [Request('long', 8, 12)] + [Request(f's{i}', 8, 2) for i in range(5)]
static = simulate_static_batching(reqs, max_batch=3)
cont = simulate_continuous_batching(reqs, max_batch=3)
print(f'static:     {len(static):2d} iters, mean batch {throughput(static):.2f}')
print(f'continuous: {len(cont):2d} iters, mean batch {throughput(cont):.2f}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.step(range(len(static)), static, where='post', label='static')
ax.step(range(len(cont)), cont, where='post', label='continuous')
ax.set_xlabel('iteration'); ax.set_ylabel('live sequences in batch')
ax.set_title('Continuous batching keeps slots full'); ax.legend()
fig.tight_layout(); plt.show()

## Exercise

Double `block_size` to 32 in the paging cell. Do you expect more or less wasted space per sequence? Run it and check against the solution.

In [ ]:
# Solution: bigger blocks -> coarser allocation -> MORE potential waste per sequence
for bs in (8, 16, 32):
    c = PagedKVCache(total_blocks=256, block_size=bs)
    c.append('x', 17)  # 17 tokens
    print(f'block_size={bs:2d}: {c.wasted_slots} wasted slots for a 17-token sequence')

## Source lab

Find the KV-cache block manager and the scheduler in the vLLM core that vLLM-Omni reuses for AR stages. Trace how a finished sequence's blocks return to the pool.